# Skeletonization and Vascular Tree Reconstruction

This notebook demonstrates skeletonization using skimage and kimimaro, followed by tree reconstruction using continuity methods.

## Setup

In [ ]:
import sys
import json
from pathlib import Path
import random
import numpy as np
import nibabel as nib
from scipy import ndimage
from skimage.morphology import skeletonize
import io

sys.path.insert(0, str(Path.cwd()))
from reconstruction import (
    load_thinning,
    estimate_point_radii_from_mask,
    LabelConnectedMultiTreeReconstruction,
    MultiTreeReconstruction,
)

try:
    import kimimaro
    HAS_KIMIMARO = True
except ImportError:
    HAS_KIMIMARO = False
    print("Warning: kimimaro not available")

## 1. Load and Merge a Random Case

In [ ]:
filled_dir = Path("data/0_test_nifti_filled")
case_dirs = sorted(d for d in filled_dir.iterdir() if d.is_dir())
case = random.choice(case_dirs)
print(f"Selected case: {case.name}\n")

nifti_files = sorted(case.glob("*.nii.gz"))
print(f"Loading {len(nifti_files)} class files and merging...")

merged = None
affine = None
for nii_path in nifti_files:
    img = nib.load(nii_path)
    if merged is None:
        merged = img.get_fdata().astype(bool)
        affine = img.affine
    else:
        merged |= img.get_fdata().astype(bool)

print(f"Merged volume shape: {merged.shape}, voxels: {merged.sum()}\n")

## 2. Skeletonize with skimage and kimimaro

In [ ]:
print("Skeletonizing...")
skimage_skel = skeletonize(merged, method="lee")
print(f"  skimage (Lee): {skimage_skel.sum()} voxels")

if HAS_KIMIMARO:
    skel_dict = kimimaro.skeletonize(
        merged.astype(np.uint8),
        teasar_params={"scale": 1.5, "const": 300},
    )
    kimimaro_skel = np.zeros_like(merged, dtype=bool)
    for skel in skel_dict.values():
        for pt in skel.vertices:
            pt_int = tuple(int(round(c)) for c in pt)
            if all(0 <= p < s for p, s in zip(pt_int, merged.shape)):
                kimimaro_skel[pt_int] = True
    print(f"  kimimaro (TEASAR): {kimimaro_skel.sum()} voxels")
else:
    kimimaro_skel = None

## 3. Extract Skeleton Points

In [ ]:
print("\nExtracting skeleton points...")

skimage_pts = np.argwhere(skimage_skel)
print(f"  skimage: {len(skimage_pts)} points")

if kimimaro_skel is not None:
    kimimaro_pts = np.argwhere(kimimaro_skel)
    print(f"  kimimaro: {len(kimimaro_pts)} points")

## 4. Reconstruct Using Continuity Methods

In [ ]:
def reconstruct_and_report(pts, mask, name):
    """Reconstruct a skeleton and report results."""
    try:
        print(f"\n{name} ({len(pts)} points)")
        print("-" * 70)

        radii = estimate_point_radii_from_mask(mask, pts, spacing=None)
        print(f"  Point radii: mean={radii.mean():.2f}, min={radii.min():.2f}, max={radii.max():.2f}")

        # Label-connected reconstruction
        lc = LabelConnectedMultiTreeReconstruction(
            skeleton_points=pts,
            mask=mask,
            point_radii=radii,
            n_trees=3,
        )
        lc.reconstruct_multiple_trees(verbose=False)
        n_edges_lc = sum(t["tree"].number_of_edges() for t in lc.trees)
        n_nodes_lc = sum(t["tree"].number_of_nodes() for t in lc.trees)
        print(f"  Label-connected: {n_edges_lc} edges, {n_nodes_lc} nodes across {len(lc.trees)} trees")

        # Basic multi-tree reconstruction
        mt = MultiTreeReconstruction(
            skeleton_points=pts,
            n_trees=3,
        )
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()
        mt.reconstruct_multiple_trees()
        sys.stdout = old_stdout

        n_edges_mt = sum(t["tree"].number_of_edges() for t in mt.trees)
        n_nodes_mt = sum(t["tree"].number_of_nodes() for t in mt.trees)
        print(f"  Basic multi-tree: {n_edges_mt} edges, {n_nodes_mt} nodes across {len(mt.trees)} trees")

        return {"lc": lc, "mt": mt}
    except Exception as e:
        print(f"  Error: {e}")
        import traceback
        traceback.print_exc()
        return None

print("\n" + "="*70)
print("RECONSTRUCTION RESULTS")
print("="*70)

skimage_results = reconstruct_and_report(skimage_pts, merged, "skimage skeleton")
if kimimaro_skel is not None:
    kimimaro_results = reconstruct_and_report(kimimaro_pts, merged, "kimimaro skeleton")
else:
    kimimaro_results = None

## 5. Summary

In [ ]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Case: {case.name}")
print(f"Merged volume: {merged.shape}, {merged.sum()} voxels")
print(f"Skeleton methods: skimage (Lee), {('kimimaro (TEASAR)' if HAS_KIMIMARO else 'N/A')}")
print(f"Reconstruction methods: label-connected, basic multi-tree")
print("\nResults available as skimage_results and kimimaro_results (if available)")